# 03 - Baselines
**Milestone 3 (spec 3.3)** | Owner: Baseline and Analysis Engineer (role 3). Needs `splits.csv` from notebook 02 (it rebuilds it if missing).
All numbers here are on the **validation** set. The test set stays untouched until the final evaluation.

In [ ]:
import os, sys, time
if os.path.exists('/content'):
    from google.colab import drive
    drive.mount('/content/drive')
    %cd /content/drive/MyDrive/aie231-project/notebooks
sys.path.insert(0, os.path.abspath('..'))
!pip -q install kagglehub
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay
from src.data import *
from src.evaluate import binary_metrics
from src.train import log_run
np.random.seed(SEED)
LOG = '../experiments/experiment_log.csv'
AUTHOR = 'baseline-engineer'

In [ ]:
ROOT = find_data_root(download())
if os.path.exists('splits.csv'):
    df = pd.read_csv('splits.csv')
    df = rebase_paths(df, ROOT)      # saved paths are stale in a new Colab session
else:
    inv = build_inventory(ROOT); inv.to_csv('inventory.csv', index=False)
    df = make_splits(drop_exact_duplicates(inv)); check_no_leakage(df); df.to_csv('splits.csv', index=False)
tr, va = df[df.split == 'train'], df[df.split == 'val']
print(len(tr), 'train |', len(va), 'val')

## Features for the classical baselines
Each X-ray is shrunk to **64x64 grey pixels** and flattened to a 4,096-number vector in [0,1]. A StandardScaler is **fit on train only**, then applied to val (no leakage). These baselines ignore image structure on purpose: they show what you get *without* convolutions, so the CNN has something to beat.

In [ ]:
t0 = time.time()
Xtr, Xva = load_flat_arrays(tr, 64), load_flat_arrays(va, 64)
ytr, yva = tr.label.values, va.label.values
scaler = StandardScaler().fit(Xtr)
Xtr_s, Xva_s = scaler.transform(Xtr), scaler.transform(Xva)
print('features', Xtr_s.shape, 'loaded in', round(time.time() - t0), 's')

## Baseline 1: majority class (trivial)
Always predict the most common training class. For this data that is PNEUMONIA, so recall(pneumonia) = 1.0 but recall(normal) = 0 and the model is useless. It shows why **accuracy alone is misleading**: its accuracy is just the pneumonia share.

In [ ]:
majority = int(pd.Series(ytr).mode()[0])
p_major = np.full(len(yva), float(majority))
m_major = binary_metrics(yva, p_major)
log_run(LOG, AUTHOR, 'baseline_majority', {'predict': majority}, SEED, 0, 0, 'val', m_major)
pd.Series(m_major)

## Baseline 2a: logistic regression
One linear layer + sigmoid: `P(pneumonia) = sigmoid(w . x + b)`, trained with log-loss. It is a neural network with no hidden layer. `C` controls L2 regularization (smaller = stronger).

In [ ]:
t0 = time.time()
lr = LogisticRegression(C=0.01, max_iter=2000, random_state=SEED).fit(Xtr_s, ytr)
p_lr = lr.predict_proba(Xva_s)[:, 1]
m_lr = binary_metrics(yva, p_lr)
log_run(LOG, AUTHOR, 'baseline_logreg', {'C': 0.01, 'features': '64x64 grey'}, SEED, time.time() - t0, Xtr_s.shape[1] + 1, 'val', m_lr)
pd.Series(m_lr)

## Baseline 2b: single-hidden-layer MLP (Keras)
`Dense(64, relu) -> Dropout(0.3) -> Dense(1, sigmoid)`, binary cross-entropy, Adam (lr 1e-3), early stopping on val loss. Parameters by hand: hidden = 4096 x 64 + 64 = 262,208; output = 64 x 1 + 1 = 65; total 262,273 (check with `model.summary()`).

In [ ]:
import tensorflow as tf
tf.keras.utils.set_random_seed(SEED)
mlp = tf.keras.Sequential([
    tf.keras.layers.Input((Xtr_s.shape[1],)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(1, activation='sigmoid')])
mlp.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='binary_crossentropy', metrics=['accuracy'])
mlp.summary()
t0 = time.time()
hist = mlp.fit(Xtr_s, ytr, validation_data=(Xva_s, yva), epochs=50, batch_size=64, verbose=2,
               callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)])
p_mlp = mlp.predict(Xva_s, verbose=0).ravel()
m_mlp = binary_metrics(yva, p_mlp)
log_run(LOG, AUTHOR, 'baseline_mlp1', {'hidden': 64, 'dropout': 0.3, 'lr': 1e-3, 'batch': 64}, SEED, time.time() - t0, mlp.count_params(), 'val', m_mlp)
pd.Series(m_mlp)

## Compare the three baselines (validation set)

In [ ]:
cmp = pd.DataFrame({'majority': m_major, 'logreg': m_lr, 'mlp_1hidden': m_mlp}).T
display(cmp[['accuracy','recall_pneumonia','recall_normal','precision_pneumonia','f1_pneumonia','macro_f1','auc']].round(3))
fig, ax = plt.subplots(1, 3, figsize=(14, 4))
for a, (name, p) in zip(ax[:2], [('logreg', p_lr), ('mlp', p_mlp)]):
    ConfusionMatrixDisplay.from_predictions(yva, (p >= .5).astype(int), display_labels=CLASSES, ax=a, colorbar=False); a.set_title(name)
for name, p in [('logreg', p_lr), ('mlp', p_mlp)]:
    RocCurveDisplay.from_predictions(yva, p, name=name, ax=ax[2])
ax[2].plot([0, 1], [0, 1], 'k--'); ax[2].set_title('ROC (val)')
plt.tight_layout(); plt.show()
pd.read_csv(LOG)

## What to write in the report (section 4.2)
* The **numbers to beat**: macro-F1, recall(pneumonia) AND recall(normal), AUC of the two learned baselines. Every neural model is reported as an improvement over these.
* The majority baseline looks good on accuracy and recall(pneumonia) but is worthless for Normal: justification for the metric choice.
* If the MLP does not beat logistic regression, say so and explain (flattened pixels throw away spatial structure; overfitting with 4,096 inputs): that is a finding, not a failure.
* Every run is in `experiments/experiment_log.csv` (spec 3.5).